# SnowflakeのMLランタイムを自前コンテナに置き換える——CREで再現性とコンプライアンスを同時に解決 — 検証ノートブック

## このノートブックについて

Zenn 記事「[SnowflakeのMLランタイムを自前コンテナに置き換える——CREで再現性とコンプライアンスを同時に解決](https://zenn.dev/gtk0326/articles/i67-feature-update-2026-05-19-custom-runtime-ima)」のハンズオン検証コードです。

記事と同じ手順を自分の Snowflake 環境で再現できます。

---

> **注意**: 各セルは上から順番に実行してください。最後のクリーンアップセルを必ず実行し、検証用オブジェクトを削除してください。

## ステップ1: Image Repository を作成する

:::details DB・スキーマ・Image Repository 作成 DDL


In [ ]:
-- データベースとスキーマを作成
CREATE DATABASE IF NOT EXISTS cre_demo_db;
CREATE SCHEMA IF NOT EXISTS cre_demo_db.ml_schema;

-- Image Repository を作成（コンテナイメージの格納場所）
CREATE IMAGE REPOSITORY cre_demo_db.ml_schema.my_image_repo;

-- レジストリ URL を確認する
SHOW IMAGE REPOSITORIES IN SCHEMA cre_demo_db.ml_schema;

## ステップ3: CRE を登録する


In [ ]:
-- CRE はアカウントレベルのオブジェクト（DB・スキーマ指定は不要）
CREATE CUSTOM RUNTIME ENVIRONMENT ml_lightgbm_env
  IMAGE_PATH = 'cre_demo_db/ml_schema/my_image_repo/ml_lightgbm_env:v1.0'
  BASE_IMAGE_TYPE = 'ML_RUNTIME';

## ステップ4: CRE の一覧を確認する


In [ ]:
-- アカウント内の CRE 一覧（スキーマ指定なし）
SHOW CUSTOM RUNTIME ENVIRONMENTS;

## ステップ5: アクセス権を付与する


In [ ]:
-- データサイエンティストロールに Image Repository への読み取り権を付与
GRANT READ ON IMAGE REPOSITORY cre_demo_db.ml_schema.my_image_repo
  TO ROLE data_scientist_role;

-- CRE の使用権を付与
GRANT USAGE ON CUSTOM RUNTIME ENVIRONMENT ml_lightgbm_env
  TO ROLE data_scientist_role;

## ステップ6: CRE を指定して Notebook を実行する


In [ ]:
-- RUNTIME パラメータで CRE を参照する
EXECUTE NOTEBOOK PROJECT mydb.myschema.analysis_notebook
  RUNTIME = 'cre@ml_lightgbm_env';

## クリーンアップ

検証で作成したオブジェクトをすべて削除してください。

> **必ず実行してください。** Dynamic Table などを残すとバックグラウンドでリフレッシュが継続しクレジットが消費されます。